In [1]:
# notebooks/02_model_evaluation.ipynb

# --- Etapa 3: Avaliacao e Selecao do Modelo ---
# Foco: Analisar resultados do MLflow, selecionar o melhor modelo e gerar visualizacoes.

# ============================================================
# CÉLULA 1 — Imports e setup de paths
# ============================================================
import sys
import os
import logging
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_recall_curve, auc,
    classification_report, confusion_matrix, roc_curve,
    ConfusionMatrixDisplay,
)

# Notebook está em project_root/notebooks/ → sobe um nível
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    from src.data.make_dataset import load_data, clean_data
    from src.features.build_features import split_data, create_preprocessor
except ImportError as e:
    print(f"Erro ao importar módulos de src/: {e}")
    sys.exit(1)

logging.basicConfig(
    level=logging.INFO,
    format='{"time": "%(asctime)s", "level": "%(levelname)s", "message": "%(message)s"}',
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)
logger = logging.getLogger(__name__)


In [ ]:
from sqlalchemy import create_engine
import mlflow.store.db.utils as db_utils
import os

db_path = project_root / "mlflow.db"

# Delete se existir
if os.path.exists(db_path):
    os.remove(db_path)
    print("Banco antigo deletado!")

# Recria do zero
uri = f"sqlite:///{db_path}"
engine = create_engine(uri)
db_utils._initialize_tables(engine)
print("Banco recriado com sucesso!")
print(f"Caminho: {db_path}")

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\PlayHard\\Desktop\\ML\\1º ciclo\\Projeto1-PortF\\churn-prediction\\mlflow.db'

: 

In [3]:

# ============================================================
# CÉLULA 2 — Configurações (paths + MLflow)
# ============================================================
processed_data_path = project_root / "data" / "processed" / "telco_customer_churn_cleaned.csv"
preprocessor_path   = project_root / "models" / "preprocessor.joblib"

# ------------------------------------------------------------------ #
#  FIX: mesma URI absoluta usada pelo train_model.py                  #
# ------------------------------------------------------------------ #
MLFLOW_TRACKING_URI    = f"sqlite:///{project_root / 'mlflow.db'}"
MLFLOW_EXPERIMENT_NAME = "Churn_Prediction_Advanced_Models_Optimized"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)   # <<< obrigatório antes de qualquer chamada
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
# ------------------------------------------------------------------ #

logger.info(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
logger.info(f"Experimento: {MLFLOW_EXPERIMENT_NAME}")



MlflowException: Detected out-of-date database schema (found version 7d34483879f0, but expected c3d6457b6d8a). Take a backup of your database, then run 'mlflow db upgrade <database_uri>' to migrate your database to the latest schema. NOTE: schema migration may result in database downtime - please consult your database's documentation for more detail.

In [ ]:

# ============================================================
# CÉLULA 3 — Carregar dados e pré-processador
# ============================================================
try:
    df_cleaned = pd.read_csv(processed_data_path)
    logger.info(f"Dados carregados: {df_cleaned.shape}")
except FileNotFoundError:
    logger.critical("Arquivo de dados limpos não encontrado. Execute make_dataset.py primeiro.")
    sys.exit(1)

X_train, X_test, y_train, y_test = split_data(df_cleaned)
logger.info(f"Split — X_train: {X_train.shape} | X_test: {X_test.shape}")

try:
    preprocessor = joblib.load(preprocessor_path)
    logger.info("Pré-processador carregado.")
except FileNotFoundError:
    logger.critical("Pré-processador não encontrado. Execute build_features.py primeiro.")
    sys.exit(1)

X_test_processed = preprocessor.transform(X_test)
logger.info("Dados de teste pré-processados.")


{"time": "2026-05-08T12:39:35Z", "level": "INFO", "message": "Dados carregados: (7043, 20)"}
{"time": "2026-05-08T12:39:35Z", "level": "INFO", "message": "{'event': 'data_split_success', 'X_train_shape': '(5634, 19)', 'X_test_shape': '(1409, 19)', 'message': 'Dados divididos em treino e teste.'}"}
{"time": "2026-05-08T12:39:35Z", "level": "INFO", "message": "Split — X_train: (5634, 19) | X_test: (1409, 19)"}
{"time": "2026-05-08T12:39:35Z", "level": "INFO", "message": "Pré-processador carregado."}
{"time": "2026-05-08T12:39:35Z", "level": "INFO", "message": "Dados de teste pré-processados."}


In [ ]:

# ============================================================
# CÉLULA 4 — Listar todos os runs do experimento
# ============================================================
all_runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    order_by=["metrics.roc_auc DESC"],
)

if all_runs.empty:
    print("⚠️  Nenhum run encontrado. Verifique se train_model.py foi executado com a URI correta.")
else:
    cols = ["tags.mlflow.runName", "metrics.roc_auc", "metrics.f1_score", "metrics.pr_auc"]
    print(all_runs[cols].to_string(index=False))


⚠️  Nenhum run encontrado. Verifique se train_model.py foi executado com a URI correta.


In [ ]:

# ============================================================
# CÉLULA 5 — Selecionar e carregar o melhor modelo
# ============================================================
best_model = None
best_run   = None

if not all_runs.empty:
    best_run = all_runs.iloc[0]       # já ordenado por roc_auc DESC
    run_id   = best_run["run_id"]

    logger.info(f"Melhor run: {run_id} | ROC AUC: {best_run['metrics.roc_auc']:.4f}")

    try:
        best_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
        logger.info(f"Modelo carregado com sucesso.")
    except Exception as e:
        logger.error(f"Erro ao carregar modelo: {e}")
else:
    logger.warning("Nenhum run disponível para carregar.")

{"time": "2026-05-08T12:39:35Z", "level": "WARNING", "message": "Nenhum run disponível para carregar."}


In [ ]:

# ============================================================
# CÉLULA 6 — Avaliação detalhada
# ============================================================
if best_model is None:
    print("Modelo não carregado. Verifique as células anteriores.")
else:
    print(f"\nModelo selecionado : {best_run['tags.mlflow.runName']}")
    print(f"Parâmetros         : {best_run.filter(like='params.').to_dict()}")

    y_pred  = best_model.predict(X_test_processed)
    y_proba = best_model.predict_proba(X_test_processed)[:, 1]

    print("\n--- Relatório de Classificação ---")
    print(classification_report(y_test, y_pred))

    # Matriz de Confusão
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Churn", "Churn"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Matriz de Confusão — Melhor Modelo")
    plt.tight_layout()
    plt.show()


Modelo não carregado. Verifique as células anteriores.


In [ ]:

# ============================================================
# CÉLULA 7 — Curva ROC
# ============================================================
if best_model is not None:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC AUC = {roc_auc_val:.3f}")
    plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("Taxa de Falsos Positivos")
    plt.ylabel("Taxa de Verdadeiros Positivos")
    plt.title("Curva ROC — Melhor Modelo")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================
# CÉLULA 8 — Curva Precision-Recall
# ============================================================
if best_model is not None:
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
    pr_auc_val = auc(recall_vals, precision_vals)

    plt.figure(figsize=(7, 5))
    plt.plot(recall_vals, precision_vals, color="blue", lw=2, label=f"PR AUC = {pr_auc_val:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Curva Precision-Recall — Melhor Modelo")
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.show()

    print("\n✅ Avaliação do melhor modelo concluída.")